In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.plot  # submodule is not imported by `import rasterio` alone
import torch
from IPython.display import Image  # For displaying the habitat selection GIF

# Use this to load the functions used in this notebook
# You can also 'import deepssf' and call functions as deepssf.function_name
import deepssf
from deepssf import (
    ConvJointModel,
    ModelParams,
    filter_steps_by_window,
    get_device,
    load_checkpoint,
    load_environmental_layers,
    load_head_weights,
    params_from_checkpoint,
    prepare_movement_df,
    simulate_trajectory,
)

print(f"deepssf {deepssf.__version__}")
print(f"torch {torch.__version__}")


# Setup the script

## Set random seed and the device for predicting from the model

If you have a GPU, this is where it will be specified (automatically).

In [ ]:
# Reproducibility — fix random seeds so results are the same on every run.
# Prediction is not deterministic: simulate_trajectory() samples a next location
# from the predicted probability surface at every step, so without a seed two
# runs of the same trained model give different trajectories (and a different
# heatmap).  Seeding here makes a set of simulations reproducible.
torch.manual_seed(42)
np.random.seed(42)

# DEVICE is capitalised — module-level constant (set once, never reassigned).
# get_device() returns 'cuda' if a GPU is available, 'mps' on Apple Silicon,
# or 'cpu' otherwise.  All tensors and the model are moved to this device.
DEVICE = get_device()
print(f"Device: {DEVICE}")

## Set script paths
These must point at the same data and rasters the model was trained on — the
model takes one input channel per raster band, in `LAYER_PATHS` order, so a
different set of layers (or a different order) either fails to load the weights
or feeds the wrong covariate into each filter.

`OUTPUT_DIR` is the directory a training run wrote its checkpoints to, given
here as a literal path rather than through `get_output_dir()` (which mints a
*new* dated directory).  Figures and rasters produced below are written back
into the same directory, alongside the model they came from.

*All-caps names below are module-level constants — set once here and referenced throughout the notebook without being reassigned.  Python has no built-in constant keyword, so capitalisation is the community convention (PEP 8) to signal "do not change this value".*


In [ ]:
DATA_DIR = Path("data")  # location of the bundled sample data
MAPPING_DIR = Path("mapping")  # location of the landcover mapping files
CSV_PATH = DATA_DIR / "pig_all_steps_covs_subset_20260813.csv"  # GPS tracking data, 28 individuals

# NOTE: 's2' and 's2_dir' are RESERVED keys in load_environmental_layers().
# 's2_dir' triggers the monthly-composite loader (load_s2_data), and a plain
# 's2' entry is skipped by the per-file loop — so a single multi-band GeoTIFF
# must be given any *other* name, otherwise nothing is loaded and the function
# raises "raster_transform could not be set".
LAYER_PATHS = {
    # Topographic layers (DEM, slope)
    "dem": str(MAPPING_DIR / "DEM_3112_crop.tif"),
    "slope": str(MAPPING_DIR / "slope_3112_crop.tif"),

    # AlphaEarth embeddings (2025)
    "alphaearth": str(MAPPING_DIR / "feral_pig_aoi_alphaearth_2025_crop.tif"),
}

# The training run to predict from — where its checkpoints were written by
# deepssf_piggies_train.ipynb, and where the outputs below are saved.
OUTPUT_DIR = Path("outputs/alphaearth/2026-08-17_4")
CKPT_PATH = OUTPUT_DIR / "best_model.pt"

if not CKPT_PATH.exists():
    raise FileNotFoundError(
        f"No checkpoint at {CKPT_PATH} — set OUTPUT_DIR to a directory that "
        "deepssf_piggies_train.ipynb has finished writing to."
    )
print(f"Output directory: {OUTPUT_DIR}")
print("Checkpoints:", ", ".join(sorted(p.name for p in OUTPUT_DIR.glob("*.pt"))))

# Import data
The csv file is imported with a timestamp and an x and y coordinate. In our case the data was already in a projected format that is specific to Australia (EPSG:3112), which is the same crs as the raster data which will be imported.

In [ ]:
raw_df = pd.read_csv(CSV_PATH)
print(f"Raw GPS fixes : {len(raw_df):,}")
print(f"Individuals   : {raw_df['name'].nunique()}")

# Sort by the individual ID and timestamp to ensure correct temporal order
raw_df = raw_df.sort_values(by=["name", "t1_"]).reset_index(drop=True)

# Convert the 'time' column to the correct timezone (here Australia/Darwin), then
# drop the offset.  prepare_movement_df() re-parses timestamps with utc=True, so a
# tz-aware Darwin column would be converted straight back to UTC and hour_t1 would
# be 9.5 h out — making the diel covariates wrong.  Localising to Darwin and then
# going tz-naive keeps the local clock time that the sin/cos encodings need.
raw_df["t1_"] = (
        pd.to_datetime(raw_df["t1_"], utc=True)
        .dt.tz_convert("Australia/Darwin")
        .dt.tz_localize(None)
    )

raw_df.head()

## `load_environmental_layers()`

Reads each GeoTIFF listed in LAYER_PATHS, min–max

- Scales every band to [0, 1], and returns:
    * `env_layers`       — dict mapping layer name → NumPy array: 2-D (H × W) for single-band GeoTIFFs, 3-D (bands × H × W) for multi-band ones, where each band becomes its own model input channel
    * `raster_transform` — Affine object that converts pixel (row, col) indices to the projected coordinates (easting, northing) of the raster(s), which should be in the same CRS as the GPS data.  This transform is used whenever we need to translate between pixel space and real-world coordinates.

**Two things to watch for:**

1. The keys `'s2'` and `'s2_dir'` are *reserved* for the monthly Sentinel-2 composite loader — a single GeoTIFF passed under the name `'s2'` is silently skipped (see the note in the paths cell above).
2. The rescaling uses a **single global min/max across all bands** of a file, and computes `(data - lo) / hi` rather than `(data - lo) / (hi - lo)`. Our index raster is `int16` scaled by 10 000 with ±10 000 fill values, so the loaded values span [0, 2] rather than [0, 1] and the three indices are not individually normalised. That is workable (the CNN can rescale internally) but if it hurts training, pre-scale each band yourself and save as `.npy` — non-`.tif` paths in `LAYER_PATHS` are loaded with `np.load(..., mmap_mode='r')` and left untouched, though at least one real `.tif` must remain in the dict so the transform can be set.


In [ ]:
env_layers, raster_transform = load_environmental_layers(LAYER_PATHS)
print("Layers loaded:", list(env_layers.keys()))
for name, arr in env_layers.items():
    print(f"  {name}: shape={arr.shape}, min={arr.min():.3f}, max={arr.max():.3f}")

# Each band of a multi-band layer becomes one CNN input channel, so count bands
# (not layers) here and reuse the total when setting ModelParams below.  Deriving
# it means switching rasters in LAYER_PATHS needs no other edits.
N_SPATIAL_CHANNELS = sum(1 if arr.ndim == 2 else arr.shape[0] for arr in env_layers.values())
print(f"Spatial channels: {N_SPATIAL_CHANNELS}")

## `prepare_movement_df()`
Converts a fixes dataframe (one row per GPS location) into a *step* dataframe (one row per movement between consecutive fixes).

- For each step it computes:
    * `dx`, `dy`         — displacement in projected CRS units (metres)
    * `bearing`          — direction of travel (radians; 0 = north, clockwise positive)
    * `bearing_tm1`      — bearing of the *previous* step, used as a model input so the network can capture directional persistence by estimating a turning angle
    * `dt_hour`          — time elapsed between consecutive fixes (hours)
    * `hour_t1`, `yday_t1` — decimal hour-of-day and day-of-year at the step start location
    * `*_sin1` / `*_cos1`  — sine/cosine encodings of hour and yday (see SCALAR_COLS below)

The trailing underscore on column names (x1_, y1_, t1_, etc.) flags them as "start of step" variables; x2_, y2_, t2_ are the corresponding "end of step" values, following the convention of the `amt` package [@Signer2019-fi]. The last fix for each individual is dropped because no forward step can be formed.

In [ ]:
# The CSV is already in amt step format, but prepare_movement_df() expects one row
# per *fix* and rebuilds the steps itself (it also derives bearing, dt_hour and the
# cyclic time encodings the model needs).  Point it at the step-start columns and
# let it re-form the steps from consecutive fixes.
step_df = prepare_movement_df(
    raw_df, 
    id_col="name", 
    time_col="t1_", 
    x_col="x1_", 
    y_col="y1_"
)
n_dropped = len(raw_df) - len(step_df)
print(f"Movement steps : {len(step_df):,}  ({n_dropped} dropped — last per individual)")
step_df.head()

## `SCALAR_COLS`
These are non-spatial inputs. For processing with convolutional layers, they are converted into spatial windows (rasters/arrays), and stacked with the truly spatial covariates.

The model receives a fixed-size spatial window centred on each step's start location.  To let the network express time-varying habitat preferences (e.g. different nocturnal vs diurnal use), scalar covariates are tiled into additional channels of the same H × W size, so every pixel "knows" the current time context when habitat suitability is estimated.

- `hour_t1_sin1` / `hour_t1_cos1`
    * Sine and cosine of (2π × hour / 24), encoding time-of-day as a *circular* variable.  The circular encoding means 23:00 and 01:00 are treated as adjacent — a plain hour value would have an artificial discontinuity at midnight.

- `yday_t1_sin1` / `yday_t1_cos1`
    * Same circular encoding for day-of-year (1–365), capturing seasonal variation in habitat use (e.g. dry-season vs wet-season movements).

- `dt_hour`
    * Elapsed time between consecutive fixes (hours).  Because GPS collars do not always fix at a perfectly regular rate, including dt_hour lets the movement kernel rescale expected displacement proportionally to the actual time interval rather than assuming a fixed fix rate. Note: dt_hour must be supplied at *simulation* time as well, so that `simulate_trajectory()` produces inputs consistent with training.

In [ ]:
SCALAR_COLS = ["hour_t1_sin1",
               "hour_t1_cos1", 
               "yday_t1_sin1", 
               "yday_t1_cos1", 
               "dt_hour"]

## `WINDOW_SIZE`
The size of the local availability window in cells/pixels - should match what was in the training script. 

In [ ]:
WINDOW_SIZE = 51

## `PIXEL_SIZE`
Raster resolution in CRS units (metres here).  Read straight off `raster_transform`
rather than typed in: the transform is the authority on the grid the model is
trained against, and a hand-entered value that disagrees with it silently
mis-scales the movement kernel — which is parameterised in metres — against a
grid measured in pixels.

In [ ]:
# transform.a is the cell width, transform.e the (negative) cell height
PIXEL_SIZE = abs(raster_transform.a)

if not math.isclose(abs(raster_transform.a), abs(raster_transform.e)):
    raise ValueError(
        f"Non-square pixels ({abs(raster_transform.a)} x {abs(raster_transform.e)}); "
        "the model assumes square cells."
    )
print(f"Pixel size: {PIXEL_SIZE} m  →  window {WINDOW_SIZE} px = "
      f"{WINDOW_SIZE * PIXEL_SIZE:,.0f} m across")

## `filter_steps_by_window()` 
Removes steps whose straight-line displacement exceeds half the spatial window (WINDOW_SIZE × PIXEL_SIZE / 2 metres). Such steps would place the true next location outside the predicted probability map, causing an index error in the loss function.

In [ ]:
step_df = filter_steps_by_window(
    step_df, window_size=WINDOW_SIZE, pixel_size=PIXEL_SIZE
)
print(f"Steps after window filter: {len(step_df):,}")

## `params_from_checkpoint()`
Weights only load into a model built with the *same* architecture, so predicting
from a trained model means reproducing its hyper-parameters exactly.  Rather
than re-typing the training notebook's constants here — where a habitat branch
one layer too shallow fails loudly, but a mis-set `pixel_size` quietly rescales
the movement kernel — `params_from_checkpoint()` recovers them from the
checkpoint and returns the `ModelParams` to hand to `ConvJointModel`.

`deepssf` writes the `ModelParams` dict into every checkpoint, so for a model
trained with 0.3.1 or later it is simply read back.  Older checkpoints (anything
in `outputs/` trained before then) do not have it, and the architecture is
recovered from the layer shapes instead: a conv weight is
`[out_channels, in_channels, k, k]` and a linear weight
`[out_features, in_features]`, which between them pin down every hyper-parameter
that changes a tensor shape — `input_channels`, `output_channels`,
`kernel_size`, both conv-layer counts, `dense_dim_in_all`, `dense_dim_hidden`
and `num_movement_params`.  Either way the result is checked against the saved
tensors, so a stored params dict that contradicts its own weights raises here
rather than failing later inside `load_state_dict`.

Three things it cannot get from the file, passed in below:

* `image_dim` — the habitat branch is fully convolutional, so nothing in the
  weights records the training window.  The movement branch's flattened size
  does constrain it, and that is checked, but only to within the slack that
  max-pooling's floor division allows (48–55 px all flatten to the same 288
  here).  `WINDOW_SIZE` also has to match `filter_steps_by_window()` above, and
  is the window the simulations step within.
* `pixel_size` — a property of the raster, read off `raster_transform`.
* `device` — where the model is being *run*, which has nothing to do with where
  it was trained; it defaults to this machine rather than to the recorded value.

`n_scalar_covariates` only fills the two `ModelParams` fields that no layer
reads; the scalar count that matters is folded into `input_channels`, which
comes off the weights.  The check below is the one thing the checkpoint cannot
do for us — that the covariates loaded in *this* notebook are the ones the model
was trained on.  It compares counts only, so keep `LAYER_PATHS` in the training
run's order.


In [ ]:
params = params_from_checkpoint(
    str(CKPT_PATH),
    image_dim=WINDOW_SIZE,               # not recorded in the weights
    pixel_size=PIXEL_SIZE,               # from raster_transform
    n_scalar_covariates=len(SCALAR_COLS),
    device=DEVICE,                       # this machine, not the training machine
)
for name, value in params.to_dict().items():
    print(f"  {name:26s}: {value}")

# The covariates in this notebook must be the ones the model was trained on.
# input_channels is one channel per raster band plus one per scalar covariate,
# so a changed LAYER_PATHS or SCALAR_COLS shows up here as a count mismatch.
n_expected = N_SPATIAL_CHANNELS + len(SCALAR_COLS)
if n_expected != params.input_channels:
    raise ValueError(
        f"{N_SPATIAL_CHANNELS} raster bands + {len(SCALAR_COLS)} scalars = {n_expected} "
        f"input channels, but the checkpoint was trained with {params.input_channels}. "
        "LAYER_PATHS or SCALAR_COLS differ from the training run."
    )
print(f"\n{N_SPATIAL_CHANNELS} raster bands + {len(SCALAR_COLS)} scalars = "
      f"{params.input_channels} input channels - matches the checkpoint")


## `ConvJointModel` 
The deepSSF model! A joint habitat–movement model with two heads:
1. Habitat head 
    — a small CNN applied to the spatial window (env layers + tiled scalars) that outputs a per-pixel log-suitability map.
2. Movement head 
    — a dense MLP (fully-connected layer) that takes the scalar covariates and the previous bearing, and produces parameters for a parametric movement kernel (a mixture of wrapped distributions centred on the current location).

The two log-probability surfaces are summed pixel-wise to give the joint predicted probability map for the animal's next step.

In [ ]:
model = ConvJointModel(params).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters : {n_params:,}")

## Load the trained model
`ConvJointModel` above is freshly initialised — random weights — so nothing it
predicts means anything until a checkpoint is loaded into it.  A training run
writes three:

- `best_model.pt` — the model to use for **simulation**.  Both heads from one
  epoch, so the joint likelihood is the one that was actually measured.
- `best_model_habitat.pt` / `best_model_movement.pt` — each head at its own best
  epoch, tracked across the whole run.

Mixing them is a deliberate trade: taking the habitat head from a different
epoch gives the sharpest habitat surface to *map*, but the two heads then come
from different epochs and the joint model's likelihood is no longer what either
checkpoint recorded.  Use it for the landscape predictions below, not for
simulating trajectories.

`load_checkpoint()` loads the weights in place and is strict about shapes, so a
mismatch between the constants set above and the trained architecture raises
here rather than silently predicting from a partly-random model.


In [ ]:
# The joint model — what the simulations further down should use
ckpt = load_checkpoint(str(CKPT_PATH), model)
print(f"best_model.pt          saved on '{ckpt['checkpoint_criterion']}' "
      f"= {ckpt['checkpoint_score']:.6f}")
print(f"                       trained with deepssf {ckpt.get('deepssf_version', 'unknown')}, "
      f"val_loss {ckpt['val_loss']:.6f}")

hab_ckpt = load_checkpoint(str(OUTPUT_DIR / "best_model_habitat.pt"))
print(f"best_model_habitat.pt  saved on '{hab_ckpt['checkpoint_criterion']}' "
      f"= {hab_ckpt['checkpoint_score']:.6f}")
print(f"(a uniform surface over the window scores {np.log(WINDOW_SIZE ** 2):.4f})")

# Prediction only — no gradients needed, and this turns off dropout/batchnorm
# training behaviour.
model.eval()

# To map the sharpest habitat surface instead, overwrite just that head:
# load_head_weights(str(OUTPUT_DIR / "best_model_habitat.pt"), model, "habitat")


## Convert the NumPy rasters loaded earlier into PyTorch tensors
`predict_habitat_landscape()`, `simulate_trajectory()` and
`simulate_trajectories()` all call `get_landscape(month_index)` to fetch the
current environmental layers as a list of 2-D tensors.

With static (non-seasonal) rasters the `month_index` argument is unused. For
seasonal data (e.g. monthly NDVI composites) you would index into a stack of
layers here based on the current month.


In [ ]:
# The simulation and prediction functions stack the list returned by
# get_landscape() along a new channel dimension, so every element must be a *2-D*
# tensor.  Unbind the band dimension of any multi-band layer so the channel order
# matches what MovementDataset fed the model during training.
landscape_tensors = []
for arr in env_layers.values():
    t = torch.from_numpy(np.asarray(arr).astype(np.float32))
    landscape_tensors.extend([t] if t.ndim == 2 else list(t))

print(f"Landscape channels: {len(landscape_tensors)}  (each {tuple(landscape_tensors[0].shape)})")


def get_landscape(_month_index):
    # Returns one 2-D tensor per spatial channel, in the same order as LAYER_PATHS.
    return landscape_tensors


# Habitat selection across the full landscape

## `predict_habitat_landscape()`
Everything so far has run the model on a `WINDOW_SIZE` crop centred on the
animal.  But the habitat sub-network is *fully convolutional* — four 3x3
convolutions with stride 1 and padding 1, no pooling and no dense layer — so the
same trained filters can be swept across the entire raster in a single pass.
Convolution is translation-equivariant, so the value at a pixel is exactly what
the model would have produced for a window centred on that pixel.

That gives a landscape-wide habitat selection surface.  Read it as a **relative
ranking of cells**: "if the animal were somewhere on this landscape, how
attractive is each cell". It deliberately says nothing about whether the animal
could *reach* a cell in one step — that constraint lives in the movement kernel,
which takes no part here.  The two are only combined over a local window, which
is what `simulate_next_step()` does.

- **Inputs**:
    * `model`
        - the trained model; only `model.conv_habitat` is used
    * `landscape_rasters`
        - the spatial covariates as a list of 2-D tensors, in the same order and
          on the same scaling as training — i.e. exactly what `get_landscape()`
          returns
    * `scalars`
        - the scalar covariates for the moment being predicted, in `SCALAR_COLS`
          order.  `make_simulation_inputs()` builds this row for us, so the
          encoding cannot drift from what the simulation uses.  Each value is
          broadcast to a constant layer covering the whole landscape (the same
          thing `Scalar_to_Grid_Block` does for a window — that block can't be
          reused here because its output size is fixed to `WINDOW_SIZE`)
    * `normalise`
        - subtract the log-sum-exp over the whole landscape, making the surface
          a log-probability that sums to 1 across all valid cells
    * `edge_buffer`
        - how many border pixels to mask as `NaN`.  Defaults to
          `habitat_edge_buffer(model)` — the exact width the zero-padding
          reaches, 1 pixel per conv layer, so 4 here.  Those pixels saw padding
          rather than landscape and are not comparable with the interior
    * `chunk_rows`
        - process this many raster rows at a time.  A full-landscape activation
          is several hundred MB, enough to exhaust a GPU; the chunks overlap by
          `edge_buffer` so the result is identical to a single pass

- **Outputs**: a `[height, width]` NumPy array of log-probabilities, `NaN` in the
  border.  Take `np.exp()` for probabilities.

In [ ]:
from deepssf import (
    habitat_edge_buffer,
    make_simulation_inputs,
    predict_habitat_landscape,
)

# Predict for one moment in time: 2 am on day 200.  make_simulation_inputs
# returns one row per step, so ask for a single step and take row 0 — that
# guarantees the sin/cos encoding matches what the simulation feeds the model.
PRED_YDAY, PRED_HOUR = 200, 2.0
scalars_pred = make_simulation_inputs(
    n_steps=1, starting_yday=PRED_YDAY, starting_hour=PRED_HOUR
)[0][0]
print(f"scalar covariates ({', '.join(SCALAR_COLS)}):\n  {scalars_pred}")

habitat_log = predict_habitat_landscape(
    model,
    landscape_rasters=get_landscape(0),   # static layers → month index 0
    scalars=scalars_pred,
)
print(f"\nSurface {habitat_log.shape}, masked border "
      f"{habitat_edge_buffer(model)} px")
print(f"log-probability range: {np.nanmin(habitat_log):.2f} to "
      f"{np.nanmax(habitat_log):.2f}")
print(f"probabilities sum to  : {np.nansum(np.exp(habitat_log)):.4f}")

### Plot the landscape-scale habitat selection surface
Plotted on the **log** scale.  Across 9.5 million cells the probabilities are
tiny (a uniform surface would give ~1e-7 per cell) and a handful of very
attractive cells dominate the linear range, so a linear colour scale washes the
whole map out.  The colour limits are clipped to percentiles for the same
reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Percentile limits: a few extreme cells otherwise dominate the colour range
vmin, vmax = np.nanpercentile(habitat_log, [1, 99.9])

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[0],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[0].set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, "
                  f"{PRED_HOUR:.0f}:00")

rasterio.plot.show(
    habitat_log, transform=raster_transform, ax=axes[1],
    cmap="viridis", vmin=vmin, vmax=vmax,
)
axes[1].plot(
    step_df["x1_"], step_df["y1_"],
    "r.", alpha=0.25, markersize=0.6, label="observed GPS fixes",
)
axes[1].legend(loc="upper right", markerscale=10)
axes[1].set_title("with observed locations overlaid")

for ax in axes:
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")

# One colourbar for both panels
fig.colorbar(axes[0].images[0], ax=axes, shrink=0.8, label="log probability")
plt.savefig(str(OUTPUT_DIR / "habitat_landscape.png"), dpi=120, bbox_inches="tight")
plt.show()

### How does habitat selection change through the day?
The scalar covariates are inputs to the habitat CNN, so the *same* landscape
produces a different surface at different times of day. Re-running the prediction
at different times of day shows what the model has learned.

Each surface is normalised over the landscape independently, so the panels show
where selection is concentrated at that hour, not the overall level of activity.

These surfaces **only** show the habitat selection, and not where would expect to
find pigs, as these maps do not take movement into consideration. To generate 
distribution maps we need to simulate movement trajectories. 

In [ ]:
PRED_HOURS = [0, 6, 12, 18]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
surfaces = {}
for ax, hour in zip(axes.flat, PRED_HOURS, strict=True):
    scalars_h = make_simulation_inputs(
        n_steps=1, starting_yday=PRED_YDAY, starting_hour=float(hour)
    )[0][0]
    surf = predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    surfaces[hour] = surf

    lo, hi = np.nanpercentile(surf, [1, 99.9])
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax, cmap="viridis", vmin=lo, vmax=hi
    )
    ax.set_title(f"{hour:02d}:00")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")

plt.suptitle(f"Habitat selection through the day (yday {PRED_YDAY})")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "habitat_landscape_by_hour.png"), dpi=120)
plt.show()

# How much does the surface actually move between hours?  If these differences
# are ~0 the model has not learned any diel variation in habitat use.
ref = surfaces[PRED_HOURS[0]]
for hour in PRED_HOURS[1:]:
    diff = np.nanmax(np.abs(surfaces[hour] - ref))
    print(f"max |log-prob difference| between 00:00 and {hour:02d}:00: {diff:.3f}")

### Create a GIF of the hourly habitat selection surfaces

Save each hourly habitat selection map as a PNG (within a subdirectory called 'habitat_selection_maps'), and then combine them into a GIF. 

In [ ]:
HABITAT_MAP_DIR = OUTPUT_DIR / "habitat_selection_maps"
HABITAT_MAP_DIR.mkdir(exist_ok=True)

# Predict all 24 hours first, then draw them.  The two passes are what lets every
# frame share one colour scale: with per-frame percentile limits (as in the 2x2
# figure above) each surface is re-normalised to its own range, so the animation
# flickers without showing where selection has actually moved.  At this raster
# size a surface is ~9 MB, so a full day is ~215 MB in memory.
hourly_surfaces = {}
for hour in range(24):
    scalars_h = make_simulation_inputs(
        n_steps=1, starting_yday=PRED_YDAY, starting_hour=float(hour)
    )[0][0]
    hourly_surfaces[hour] = predict_habitat_landscape(
        model, landscape_rasters=get_landscape(0), scalars=scalars_h
    )
    print(f"  {hour:02d}:00 done", end="\r")

# Shared colour limits over the whole day, clipped to the same percentiles used
# for the single-hour map (a handful of extreme cells otherwise take the range).
VMIN = min(np.nanpercentile(s, 1) for s in hourly_surfaces.values())
VMAX = max(np.nanpercentile(s, 99.9) for s in hourly_surfaces.values())
print(f"shared colour limits: {VMIN:.2f} to {VMAX:.2f}")

for hour, surf in hourly_surfaces.items():
    fig, ax = plt.subplots(figsize=(15, 12))
    rasterio.plot.show(
        surf, transform=raster_transform, ax=ax,
        cmap="viridis", vmin=VMIN, vmax=VMAX,
    )
    ax.set_title(f"Habitat selection (log prob) — yday {PRED_YDAY}, {hour:02d}:00")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    fig.colorbar(ax.images[0], ax=ax, shrink=0.8, label="log probability")

    # Zero-padded hour so the filenames sort into clock order.  Note the absence
    # of bbox_inches="tight": it trims each figure to its own content, and frames
    # of differing pixel size cannot be stacked into a GIF.
    fig.savefig(HABITAT_MAP_DIR / f"habitat_hour_{hour:02d}.png", dpi=110)
    plt.close(fig)   # close, or 24 figures are drawn into the notebook output

print(f"Wrote 24 frames to {HABITAT_MAP_DIR}")

In [ ]:
import imageio.v2 as imageio

habitat_gif = OUTPUT_DIR / "habitat_selection_hourly.gif"

# Read the frames back off disk, sorted by filename — the zero-padded hour in the
# name puts them in clock order.  create_gif() (used for the training snapshots
# above) is deliberately not used here: it orders frames by an `epoch_<n>` field
# in the filename and, with no such field to key on, would leave them in whatever
# order the filesystem happened to list them.
frame_paths = sorted(HABITAT_MAP_DIR.glob("habitat_hour_*.png"))
frames = [imageio.imread(p) for p in frame_paths]

# duration is milliseconds *per frame*: half a second an hour 
# (`fps=` is accepted but deprecated in imageio v2 and warns.)
# loop=0 means loop forever rather than play once.
imageio.mimsave(habitat_gif, frames, duration=200, loop=0)
print(f"{len(frames)} frames → {habitat_gif} "
      f"({habitat_gif.stat().st_size / 1e6:.1f} MB)")

# As with the training GIF, Image() embeds the bytes so it renders wherever the
# notebook is opened.
Image(filename=str(habitat_gif))

### Save the hourly habitat selection surfaces as a raster (TIF)
Writing them out with the raster's transform and CRS means they can be opened
directly in QGIS/ArcGIS or read back with `rasterio`, and overlaid on the other
layers in `mapping/`.

One **band per hour** rather than one file per hour: the 24 surfaces share a
grid, a transform and a CRS, so a single stack keeps them together and lets GIS
software step through the day with the band slider.  Band *n* is hour *n - 1*
(GDAL numbers bands from 1), and each band carries its hour in both its
description and its tags — so `src.read(15)` is 14:00.


In [ ]:
# The CRS is not returned by load_environmental_layers(), so read it off the
# source raster the transform came from.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

# Reuses the surfaces predicted for the GIF above, so run that cell first.
# Sorting the keys rather than trusting dict insertion order means band n is
# always hour n - 1, however the surfaces were generated.
hours_sorted = sorted(hourly_surfaces)
height, width = hourly_surfaces[hours_sorted[0]].shape

habitat_tif = OUTPUT_DIR / f"habitat_landscape_yday{PRED_YDAY}_hourly.tif"
with rasterio.open(
    habitat_tif, "w",
    driver="GTiff",
    height=height,
    width=width,
    count=len(hours_sorted),      # one band per hour
    dtype="float32",
    crs=raster_crs,
    transform=raster_transform,
    nodata=np.nan,          # the masked edge buffer
    compress="deflate",
) as dst:
    for band, hour in enumerate(hours_sorted, start=1):   # GDAL bands are 1-indexed
        dst.write(hourly_surfaces[hour].astype("float32"), band)
        # The description is what QGIS shows in its band dropdown; the tags keep
        # the hour machine-readable for anything parsing the metadata.
        dst.set_band_description(band, f"habitat selection log prob {hour:02d}:00")
        dst.update_tags(band, hour=hour, yday=PRED_YDAY)
    dst.update_tags(prediction_yday=PRED_YDAY, hours=",".join(str(h) for h in hours_sorted))

print(f"Wrote {habitat_tif}")
print(f"  {len(hours_sorted)} bands, {width} x {height} px "
      f"({habitat_tif.stat().st_size / 1e6:.1f} MB)")

# Simulate trajectories

Select starting parameters for the model.

We've set the starting location to be the first location of the observed data.

In [ ]:
# Where and when the simulations start.  Taking the time from the same observed
# step as the location keeps the two consistent — the diel and seasonal
# covariates then describe a moment the animal really was at that place.
START_IDX = 10000
start_x = step_df["x1_"].iloc[START_IDX]
start_y = step_df["y1_"].iloc[START_IDX]
START_YDAY = step_df["yday_t1"].iloc[START_IDX]
START_HOUR = step_df["hour_t1"].iloc[START_IDX]
print(f"Starting location: ({start_x:.0f}, {start_y:.0f})")
print(f"Starting time    : day {START_YDAY:.0f}, {START_HOUR:.2f} h")


# Simulate a trajectory

## `simulate_trajectory()` 
Generates a sequence of steps by iteratively:
1. Extracting a spatial window centred on the current location.
2. Running the model to obtain a joint log-probability surface over that window.
3. Sampling the next pixel from that surface (softmax-normalised probabilities).
4. Converting the sampled pixel back to projected coordinates and advancing time.

- **Inputs**: 
    * `model`
        - the trained model
    * `get_landscape`
        - retrieves the landscape layers (spatial covariates) that were used to train the model
    * `transform`
        - raster Affine transform
    * `start_x`
        - x coordinate of starting location
    * `start_y`
        - y coordinate of starting location
    * `n_steps`
        - number of steps
    * `starting_yday`
        - starting day-of-year 
    * `starting_hour`
        - starting hour-of-day
    * `time_between_steps`
        - time between steps in hours
    * `window_size`
        - size in pixels of the local landscape
    * `month_index_fn`
        - a function mapping day-of-year → month index (used by the `get_landscape` function). Here we are using static spatial layers so we can set it to 0.

- **Outputs**: DataFrame with one row per simulated step containing:
    * `x`, `y`           
        — projected coordinates of the simulated location
    * `hour`, `yday`     
        — time variables updated at each step
    * `month_index`
        - the month index used to index the spatial variables (will be 0 if using static spatial covariates)

In [ ]:
sim_df = simulate_trajectory(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=1000,
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    time_between_steps=1.0,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"Simulated {len(sim_df)} steps")
sim_df.head()

### Plot the simulated data

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
# First spatial channel (NDVI with the index raster) as the background
rasterio.plot.show(
    landscape_tensors[2].numpy(), transform=raster_transform, ax=ax, cmap="viridis"
)
ax.plot(
    step_df["x1_"], step_df["y1_"],
    "w.", alpha=0.3, markersize=0.7, label="observed",
)
ax.plot(
    sim_df["x"], sim_df["y"],
    "r-o", markersize=1, linewidth=0.8, label="simulated",
)
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.legend(markerscale=6)
ax.set_title("Simulated vs observed pig locations")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectory.png"), dpi=100)
plt.show()

# Plot with an interactive map

## `plot_trajectories_folium()`
A static figure is fine for checking the overall shape of a trajectory, but not for
asking *where* the animal actually went.  This draws the simulated (and optionally
observed) locations over a satellite basemap so you can zoom in on individual steps.

- **Inputs**:
    * `trajectories`
        - a simulated-locations DataFrame.  If it has a `trajectory_id` column
          (as `simulate_trajectories()` returns) each id is drawn as its own
          toggleable line; a plain `sim_df` is drawn as a single line.
    * `observed`
        - observed locations to overlay, e.g. `step_df`.  Drawn as points rather
          than a line by default, because `step_df` holds 28 individuals and a
          single line would connect them with jumps that never happened.
    * `src_crs`
        - CRS of the input coordinates.  Everything is reprojected to EPSG:4326
          (lat/lon) for the web map — our data is in EPSG:3112.
    * `basemap`
        - `"esri_imagery"` (default), `"esri_topo"`, `"openstreetmap"`, or any
          XYZ tile-URL template.  See the folium docs for more:
          <https://python-visualization.github.io/folium/latest/user_guide/raster_layers/tiles.html>
    * `max_observed_points`
        - observed locations are thinned to this many points.  folium renders
          every point as its own element in the page, so all 100k+ fixes would
          lock up the notebook.
    * `save_path`
        - optionally also write a standalone HTML file.

- **Outputs**: a `folium.Map` — leave it as the last line of a cell to display it.

`folium` is an optional dependency: `pip install folium` (or
`pip install "deepSSF[maps]"`).

In [ ]:
from deepssf import plot_trajectories_folium

fmap = plot_trajectories_folium(
    sim_df,
    observed=step_df,
    src_crs="EPSG:3112",          # projected CRS of the GPS data and rasters
    basemap="esri_imagery",
    zoom_start=12,
    max_observed_points = 100000,
    save_path=str(OUTPUT_DIR / "simulated_trajectory_map.html"),
)
fmap

# Save the trajectory to a CSV

## `save_trajectories()`
Writes the simulated locations to `outputs/deepSSF_trajectories/`.  The filename
records how many trajectories, how many steps and the date, and the function
**never overwrites an existing file** — re-running the cell appends a `_2`, `_3`,
... run counter instead, so repeated runs accumulate rather than clobber.

- **Inputs**:
    * `trajectories` — a DataFrame (or list of DataFrames)
    * `output_dir` — created if it doesn't exist
    * `prefix` — leading part of the filename
    * `split` — write one file per `trajectory_id` instead of one combined file
- **Outputs**: the list of paths written

In [ ]:
from deepssf import save_trajectories

TRAJ_DIR = OUTPUT_DIR / "deepSSF_trajectories"

save_trajectories(sim_df, TRAJ_DIR, prefix="deepSSF_pigs")

# Simulating many trajectories

## `simulate_trajectories()`
Runs `n_trajectories` simulations and returns them stacked in one long-format
DataFrame with `trajectory_id` and `step` columns.

Three `method`s are available:

| `method` | how it works | when to use it |
| --- | --- | --- |
| `"batched"` *(default)* | every trajectory takes its step together, so the model sees a batch of `n_trajectories` windows per forward pass instead of one | almost always — the forward pass is the whole cost of the simulation, and it is far better used at batch size 50 than at batch size 1 |
| `"sequential"` | a plain loop over `simulate_trajectory()` | reference behaviour; needed if trajectories start at *different times* |
| `"parallel"` | the same loop across a thread pool | when start times vary and you want some overlap; the speed-up is modest (PyTorch releases the GIL inside its ops, but torch is already using several threads per op) and results are not reproducible from a seed |

The one restriction on `"batched"` is that all trajectories share a clock, so
`starting_yday` / `starting_hour` must be scalars.  Start *locations* can still
differ per trajectory — pass a list to `start_x` / `start_y` to release simulated
animals from different points.

In [ ]:
from deepssf import simulate_trajectories

N_TRAJECTORIES = 1000
N_SIM_STEPS = 1000

sim_many = simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=start_x,
    start_y=start_y,
    n_steps=N_SIM_STEPS,
    n_trajectories=N_TRAJECTORIES,
    method="batched",
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    time_between_steps=1.0,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,   # static layers
)
print(f"{sim_many['trajectory_id'].nunique()} trajectories x "
      f"{N_SIM_STEPS} steps = {len(sim_many):,} locations")
sim_many.head()

### Releasing simulated animals from different starting points
Passing a sequence to `start_x` / `start_y` gives each simulated animal its own
release site.  Here the release sites come from an aerial survey: each surveyed
group contributes one trajectory per pig counted, so the simulated population
matches the surveyed one — the same batched forward pass, so it costs nothing extra.


In [ ]:
# Draw starting locations from the observed fixes
# release = step_df.sample(N_TRAJECTORIES, random_state=42)

# Draw starting locations from aerial survey data (recorded in WGS84 lat/lon)
import rasterio.warp  # submodule is not imported by `import rasterio` alone

survey = pd.read_csv(DATA_DIR / "KNP SS 250809_survey_pig.csv")

# Reproject to the rasters' CRS — read off the source raster the transform came from
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs
    survey["x"], survey["y"] = rasterio.warp.transform(
    "EPSG:4326", raster_crs,
    survey["longitude"].to_numpy(), survey["latitude"].to_numpy(),
)

# Keep only sightings inside the prediction extent (the covariate rasters)
left, bottom, right, top = rasterio.transform.array_bounds(
    *landscape_tensors[0].shape, raster_transform
)

inside = survey["x"].between(left, right) & survey["y"].between(bottom, top)
print(f"{inside.sum()} of {len(survey)} survey sightings inside the raster extent")
survey = survey[inside]

# One trajectory per pig: repeat each sighting 'count' times
release = survey.loc[survey.index.repeat(survey["count"].astype(int))]
N_TRAJECTORIES = len(release)

# Start the simulations at the first survey timestamp (local clock time, as the
# model's diel/seasonal covariates were trained on)
t0 = pd.to_datetime(survey["timestamp"]).min()
START_YDAY = t0.dayofyear
START_HOUR = t0.hour + t0.minute / 60

print(f"{N_TRAJECTORIES} pigs at {len(survey)} release sites")
print(f"Starting time: day {START_YDAY}, {START_HOUR:.2f} h ({t0})")

release.head()

### How many trajectories per surveyed pig?

In [ ]:
N_ITERATIONS = 1       # how many simulations per pig, per starting location
N_SIM_STEPS =  100000    # updating here for longer trajectories

print(f"Simulating {N_ITERATIONS} trajectories per pig, "
      f"{N_SIM_STEPS} steps each, {N_TRAJECTORIES} pigs = "
      f"{N_ITERATIONS * N_SIM_STEPS * N_TRAJECTORIES:,} locations")

### Simulate from the aerial survey locations

In [ ]:
sim_scattered = simulate_trajectories(
    model,
    get_landscape=get_landscape,
    transform=raster_transform,
    start_x=release["x"].to_numpy(),
    start_y=release["y"].to_numpy(),
    n_steps=N_SIM_STEPS,
    n_trajectories=N_TRAJECTORIES * N_ITERATIONS,
    method="batched",
    starting_yday=START_YDAY,
    starting_hour=START_HOUR,
    window_size=WINDOW_SIZE,
    month_index_fn=lambda _yday: 0,
)
print(f"{len(sim_scattered):,} locations from {N_TRAJECTORIES * N_ITERATIONS} release sites")


### Plot all of the simulated trajectories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, df, title in zip(
    axes,
    [sim_many, sim_scattered],
    ["Same release site", "Aerial survey release sites"],
    strict=True,
):
    rasterio.plot.show(
        landscape_tensors[2].numpy(), transform=raster_transform, ax=ax, cmap="viridis"
    )
    ax.plot(
        step_df["x1_"], step_df["y1_"],
        "w.", alpha=0.2, markersize=0.5, label="observed",
    )
    # One colour per simulated individual
    for tid, sub in df.groupby("trajectory_id"):
        ax.plot(sub["x"], sub["y"], "-", linewidth=0.7, alpha=0.9, label=f"sim {tid}")
    ax.set_xlabel("Easting (m, projected)")
    ax.set_ylabel("Northing (m, projected)")
    ax.set_title(title)

plt.suptitle(f"{N_TRAJECTORIES} simulated trajectories")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_trajectories.png"), dpi=100)
plt.show()

### The same trajectories on the interactive map
Each trajectory gets its own colour and its own entry in the layer control (top
right), so individual animals can be switched on and off.

In [ ]:
# folium writes every trajectory into the page as its own feature group, so all
# 599 make a map too heavy to open.  Draw a subset — the heatmap below is what
# summarises the full set.
N_MAP_TRAJECTORIES = 10
map_ids = sim_scattered["trajectory_id"].unique()[:N_MAP_TRAJECTORIES]

plot_trajectories_folium(
    sim_scattered[sim_scattered["trajectory_id"].isin(map_ids)],
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points = 100000,
    save_path=str(OUTPUT_DIR / "simulated_trajectories_map.html"),
)


## Create a heatmap of the simulated trajectories

Count the number of simulated locations per cell/pixel, with an option to discard the first *n* steps to reduce bias towards the initial parameters.

## `trajectory_heatmap()`
* **Inputs**
    - `trajectories`
        - simulated locations, as returned by `simulate_trajectories()` (or a list of DataFrames).
    - `transform`, `shape`
        - the landscape rasters' affine transform and `(height, width)`, so the heatmap grid shares their origin and CRS.
    - `burn_in`
        - discard the first *n* steps of every trajectory.  Early steps cluster around the release site(s) and reflect the starting conditions rather than the fitted model.
    - `agg`
        - aggregate `agg` x `agg` raster pixels into one heatmap cell.  At the native 20 m almost every occupied cell holds a single location, so some coarsening makes the surface readable.
* **Outputs**: `(counts, heatmap_transform)` — an `int32` array of locations per cell, and its affine transform.

The result is visualised as a PNG below, written to a GeoTIFF with
`save_raster()` so it can be opened in QGIS, and added to the interactive map as
a layer that can be toggled on and off with `add_heatmap_overlay()`.

In [ ]:
from deepssf import trajectory_heatmap

HEATMAP_DF   = sim_scattered  # trajectories to summarise (or sim_many)
BURN_IN      = 0              # drop the first n steps of every trajectory
HEATMAP_AGG  = 4              # heatmap cell = HEATMAP_AGG x HEATMAP_AGG raster pixels

heatmap, heatmap_transform = trajectory_heatmap(
    HEATMAP_DF,
    raster_transform,
    landscape_tensors[0].shape,
    burn_in=BURN_IN,
    agg=HEATMAP_AGG,
)

print(f"Heatmap grid: {heatmap.shape[0]} x {heatmap.shape[1]} cells of "
      f"{PIXEL_SIZE * HEATMAP_AGG:.0f} m")
print(f"{heatmap.sum():,} locations counted after dropping the first {BURN_IN} steps "
      f"of each trajectory")
print(f"{(heatmap > 0).sum():,} occupied cells, up to {heatmap.max():,} locations in one cell")

### Plot the heatmap
Counts are drawn on a **log** colour scale — a handful of cells near the release
sites usually hold an order of magnitude more locations than the rest, which
would flatten a linear scale to a single bright dot.  Empty cells are masked so
the landscape shows through.

In [ ]:
from matplotlib.colors import LogNorm

fig, ax = plt.subplots(figsize=(8.5, 7))
# Greyscale background so the (colourful) heatmap stays readable on top
rasterio.plot.show(
    landscape_tensors[2].numpy(), transform=raster_transform, ax=ax, cmap="Greys_r"
)
# ax.plot(
#     step_df["x1_"], step_df["y1_"],
#     "c.", alpha=0.15, markersize=0.5, label="observed",
# )
img = ax.imshow(
    np.ma.masked_equal(heatmap, 0),  # empty cells stay transparent
    extent=rasterio.plot.plotting_extent(heatmap, heatmap_transform),
    cmap="magma",
    norm=LogNorm(vmin=1, vmax=max(int(heatmap.max()), 2)),
    interpolation="nearest",
    alpha=0.85,
)
fig.colorbar(img, ax=ax, label=f"simulated locations per {PIXEL_SIZE * HEATMAP_AGG:.0f} m cell")
ax.set_xlabel("Easting (m, projected)")
ax.set_ylabel("Northing (m, projected)")
ax.set_title(
    f"Simulated space use — {HEATMAP_DF['trajectory_id'].nunique()} trajectories, "
    f"steps {BURN_IN}+"
)
ax.legend(markerscale=8, loc="upper right")
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "simulated_heatmap.png"), dpi=150)
plt.show()

### Save the heatmap as a raster (TIF)

## `save_raster()`
Writes any landscape-shaped array out as a GeoTIFF, given the transform and CRS
that georeference it — the heatmap here, or the habitat selection surface
above.  `nodata=0` means unvisited cells are drawn as transparent rather than as
a solid zero-valued block.

In [ ]:
from deepssf import save_raster

# load_environmental_layers() does not return the CRS, so read it off the source
# raster the transform came from.
with rasterio.open(next(iter(LAYER_PATHS.values()))) as src:
    raster_crs = src.crs

heatmap_tif = save_raster(
    heatmap,
    OUTPUT_DIR
    / f"simulated_heatmap_{HEATMAP_DF['trajectory_id'].nunique()}traj"
      f"_{PIXEL_SIZE * HEATMAP_AGG:.0f}m_burnin{BURN_IN}.tif",
    transform=heatmap_transform,
    crs=raster_crs,
    nodata=0,               # cells never visited
    band_descriptions="simulated locations per cell",
)

### Toggle the heatmap on the interactive map

## `add_heatmap_overlay()`
`plot_trajectories_folium()` returns the `folium.Map`, so the heatmap can be
added to it afterwards as an image overlay.  folium writes each layer's
JavaScript in the order it was added, so `add_heatmap_overlay()` moves the layer
control back to the end — otherwise it would run before the overlay exists and
throw, taking the rest of the map's script with it.  The heatmap then picks up
its own checkbox alongside the observed locations and each simulated trajectory.

The counts are warped from the projected CRS to Web Mercator — the projection
Leaflet stretches an overlay in — and coloured with the same log scale as the
PNG, with cells below `vmin` (1 by default) left fully transparent.


In [ ]:
from deepssf import add_heatmap_overlay

# Same subset of trajectories as the map above; the overlay covers all of them.
heat_ids = HEATMAP_DF["trajectory_id"].unique()[:N_MAP_TRAJECTORIES]

fmap_heat = plot_trajectories_folium(
    HEATMAP_DF[HEATMAP_DF["trajectory_id"].isin(heat_ids)],
    observed=step_df,
    src_crs="EPSG:3112",
    basemap="esri_imagery",
    zoom_start=11,
    max_observed_points=100000,
)
add_heatmap_overlay(
    fmap_heat,
    heatmap,
    heatmap_transform,
    src_crs="EPSG:3112",
    name=f"heatmap (max {heatmap.max():,} per cell)",
    cmap="magma",
    opacity=0.75,
)

heatmap_html = OUTPUT_DIR / "simulated_heatmap_map.html"
fmap_heat.save(str(heatmap_html))
print(f"Map saved: {heatmap_html}")
fmap_heat


### Save them all
`split=False` (the default) writes one combined CSV with the `trajectory_id`
column; `split=True` writes one file per simulated individual.

In [ ]:
save_trajectories(sim_scattered, TRAJ_DIR, prefix="deepSSF_pigs")

# ...or one file per simulated animal:
# save_trajectories(sim_many, TRAJ_DIR, prefix="deepSSF_pigs", split=True)